In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import networkx as nx
from sklearn.manifold import SpectralEmbedding
import itertools
# import plotly.graph_objects as go
from typing import Set, Dict, Any, List, Tuple
from scipy import sparse
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from typing import Union, Optional
import plotly.express as px
import psutil



In [4]:
pwd

'/data/jupyter'

In [3]:
h = pd.read_csv('/data/elugos/goose_NY/goose_NY/triples/goose_20171101.export_triples.csv')

In [3]:
pd.reset_option("all")

As the xlwt package is no longer maintained, the xlwt engine will be removed in a future version of pandas. This is the only engine in pandas that supports writing in the xls format. Install openpyxl and write to an xlsx file instead.

: boolean
    use_inf_as_null had been deprecated and will be removed in a future
    version. Use `use_inf_as_na` instead.



/Users/emilylugos/opt/anaconda3/lib/python3.9/site-packages/pandas/_config/config.py:630: FutureWarning: As the xlwt package is no longer maintained, the xlwt engine will be removed in a future version of pandas. This is the only engine in pandas that supports writing in the xls format. Install openpyxl and write to an xlsx file instead.
  warnings.warn(d.msg, FutureWarning)
/Users/emilylugos/opt/anaconda3/lib/python3.9/site-packages/pandas/_config/config.py:630: FutureWarning: 
: boolean
    use_inf_as_null had been deprecated and will be removed in a future
    version. Use `use_inf_as_na` instead.

  warnings.warn(d.msg, FutureWarning)


In [ ]:
pd.set_option('expand_frame_repr', True)
pd.set_option("display.max_columns", None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

In [2]:
d_20250529 = pd.read_csv('data/triples_CO_sbert/goose_20250529.export_triples_sbert.csv')
d_20250530 = pd.read_csv('data/triples_CO_sbert/goose_20250530.export_triples_sbert.csv')
d_20250531 = pd.read_csv('data/triples_CO_sbert/goose_20250531.export_triples_sbert.csv')
d_20250601 = pd.read_csv('data/triples_CO_sbert/goose_20250601.export_triples_sbert.csv')
d_20250602 = pd.read_csv('data/triples_CO_sbert/goose_20250602.export_triples_sbert.csv')
d_20250603 = pd.read_csv('data/triples_CO_sbert/goose_20250603.export_triples_sbert.csv')
d_20250604 = pd.read_csv('data/triples_CO_sbert/goose_20250604.export_triples_sbert.csv')
d_20250605 = pd.read_csv('data/triples_CO_sbert/goose_20250605.export_triples_sbert.csv')
CO_x2d = pd.concat([d_20250529, d_20250530, d_20250531,d_20250601,d_20250602,d_20250603,d_20250604,d_20250605],ignore_index=True)


FileNotFoundError: [Errno 2] No such file or directory: 'data/triples_CO_sbert/goose_20250529.export_triples_sbert.csv'

In [3]:
d_20140521 = pd.read_csv('data/triples_CA_sbert/goose_20140521.export_triples_sbert.csv')
d_20140522 = pd.read_csv('data/triples_CA_sbert/goose_20140522.export_triples_sbert.csv')
d_20140523 = pd.read_csv('data/triples_CA_sbert/goose_20140523.export_triples_sbert.csv')
d_20140524 = pd.read_csv('data/triples_CA_sbert/goose_20140524.export_triples_sbert.csv')
d_20140525 = pd.read_csv('data/triples_CA_sbert/goose_20140525.export_triples_sbert.csv')
d_20140526 = pd.read_csv('data/triples_CA_sbert/goose_20140526.export_triples_sbert.csv')
d_20140527 = pd.read_csv('data/triples_CA_sbert/goose_20140527.export_triples_sbert.csv')
d_20140528 = pd.read_csv('data/triples_CA_sbert/goose_20140528.export_triples_sbert.csv')
CA_x2d = pd.concat([d_20140521,d_20140522,d_20140523,d_20140524,d_20140525,d_20140526,d_20140527,d_20140528],ignore_index=True)

FileNotFoundError: [Errno 2] No such file or directory: 'data/triples_CA_sbert/goose_20140521.export_triples_sbert.csv'

In [ ]:
def run_dbscan_OLD(df, eps=0.15, min_samples=5, scale=False, metric='cosine'):
    """
    Run DBSCAN clustering on a DataFrame with 'x' and 'y' columns.

    Args:
        df (pd.DataFrame): Must contain 'x' and 'y' columns.
        eps (float): DBSCAN eps parameter.
        min_samples (int): DBSCAN min_samples parameter.
        scale (bool): Whether to standardize features before clustering.

    Returns:
        df (pd.DataFrame): Original dataframe with added 'cluster' column.
        summary (dict): Dictionary with cluster count and noise count.
    """
    # 1) Extract coordinates
    # X = df[['x', 'y']].to_numpy(dtype=float)
    df_copy = df.copy(deep=True) 

    s = df['embedding_json'].str.lstrip('[')
    s = s.str.rstrip(']')
    s = s.str.split(',')
    s = s.apply(np.array)
    df_copy['embedding_json'] = s
    X = np.vstack(df_copy['embedding_json']).astype(np.float32)

    # 2) Scale if requested
    if scale:
        X = StandardScaler().fit_transform(X)

    # 3) Fit DBSCAN
    db = DBSCAN(eps=eps, min_samples=min_samples, metric=metric)
    labels = db.fit_predict(X)

    # 4) Attach results
    df_copy = df.copy()
    df_copy['cluster'] = labels
    df_copy = df_copy.drop('embedding_json', axis =1)
    # 5) Quick summary
    n_noise = (labels == -1).sum()
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    print(n_clusters)
    summary = {
        "n_clusters": n_clusters,
        "n_noise": n_noise
    }

    return X, df_copy

        # 6) Optional visualization (first two dims)



In [34]:
#CA_db = run_dbscan(CA_x2d)
X, CO_db = run_dbscan(CO_x2d)

ValueError: too many values to unpack (expected 2)

In [40]:
 
fig = px.scatter(
    x=X[:, 0], y=X[:, 1],
    color=CO_db.cluster,
    hover_data={
        "sbert": CO_db['sbert_text'],
        "cluster": CO_db['cluster'],
        "Event" :CO_db['GlobalEventID']


    },
    title="DBSCAN Clusters (first two embedding dimensions)"
)
#fig.update_traces(marker=dict(size=6, opacity=0.7))
fig.show()



In [20]:
CO_db

,GlobalEventID,date,source_index,subject,relation,object,sbert_text,cluster
0,1246717267,20250529,81,$,construct,dorms,$ construct dorms,-1
1,1246827445,20250529,231,$ 1 billion,absorb,cost of state ’s Medicaid program,$ 1 billion absorb cost of state ’s Medicaid p...,-1
2,1246813551,20250529,195,$ 100 million year,is in,cocaine,$ 100 million year is in cocaine,-1
3,1246717267,20250529,81,$ 301 million,construct,additional dorms,$ 301 million construct additional dorms,-1
4,1246729860,20250529,96,%,happened at,28 weeks,% happened at 28 weeks,-1
...,...,...,...,...,...,...,...,...
25133,1247935942,20250605,129,yourself,loved,ones,yourself loved ones,-1
25134,1248091932,20250605,316,‑,tailed grouse across,range of species,‑ tailed grouse across range of species,-1
25135,1247941114,20250605,135,’s,come out at_time,today,’s come out at_time today,-1
25136,1248114978,20250605,349,•,can provide,sense of confidence,• can provide sense of confidence,-1


In [29]:
def run_dbscan(df, eps=0.15, min_samples=5, scale=False, metric='cosine'):
    """
    Run DBSCAN clustering using vectors stored in 'embedding_json'.

    Args:
        df (pd.DataFrame): Must contain 'embedding_json' as a string like "[0.1, 0.2, ...]".
        eps (float): DBSCAN eps parameter (distance threshold).
        min_samples (int): DBSCAN min_samples parameter.
        scale (bool): Whether to standardize features before clustering.
        metric (str): Distance metric (e.g., 'cosine', 'euclidean').

    Returns:
        clustered_df (pd.DataFrame): Rows with cluster labels >= 0 (noise removed).
        noise_df (pd.DataFrame): Rows with cluster == -1.
        summary (dict): {'n_clusters': int, 'n_noise': int}
    """
    # Work on a copy
    df_work = df.copy(deep=True)

    # Parse 'embedding_json' -> np.ndarray
    s = df_work['embedding_json'].astype(str).str.strip()
    s = s.str.lstrip('[').str.rstrip(']')
    s = s.str.split(',')
    vecs = [np.array(list(map(float, parts)), dtype=np.float32) for parts in s]
    X = np.vstack(vecs)

    # Optional scaling (often unnecessary for cosine)
    if scale:
        X = StandardScaler().fit_transform(X)

    # Fit DBSCAN
    db = DBSCAN(eps=eps, min_samples=min_samples, metric=metric)
    labels = db.fit_predict(X)
    df_work['cluster'] = labels

    # Split noise vs non-noise
    noise_mask = df_work['cluster'] == -1
    noise_df = df_work.loc[noise_mask].drop(columns=['embedding_json']).reset_index(drop=True)
    clustered_df = df_work.loc[~noise_mask].drop(columns=['embedding_json']).reset_index(drop=True)

    # Summary
    unique_labels = set(labels)
    n_clusters = len(unique_labels - {-1})
    n_noise = int((labels == -1).sum())
    summary = {"n_clusters": n_clusters, "n_noise": n_noise}

    return X, clustered_df, noise_df


In [37]:
def run_dbscan(df, eps=0.15, min_samples=5, scale=False, metric='cosine'):
    """
    Run DBSCAN clustering using vectors stored in 'embedding_json'.

    Args:
        df (pd.DataFrame): Must contain 'embedding_json' as a string like "[0.1, 0.2, ...]".
        eps (float): DBSCAN eps parameter (distance threshold).
        min_samples (int): DBSCAN min_samples parameter.
        scale (bool): Whether to standardize features before clustering.
        metric (str): Distance metric (e.g., 'cosine', 'euclidean').

    Returns:
        X_clustered (ndarray): Embedding matrix for non-noise rows only.
        clustered_df (pd.DataFrame): Rows with cluster labels >= 0 (noise removed).
        noise_df (pd.DataFrame): Rows with cluster == -1.
        summary (dict): {'n_clusters': int, 'n_noise': int}
    """
    df_work = df.copy(deep=True)

    # Parse embeddings
    s = df_work['embedding_json'].astype(str).str.strip()
    s = s.str.lstrip('[').str.rstrip(']')
    s = s.str.split(',')
    vecs = [np.array(list(map(float, parts)), dtype=np.float32) for parts in s]
    X = np.vstack(vecs)

    # Optional scaling
    if scale:
        X = StandardScaler().fit_transform(X)

    # Fit DBSCAN
    db = DBSCAN(eps=eps, min_samples=min_samples, metric=metric)
    labels = db.fit_predict(X)
    df_work['cluster'] = labels

    # Masks
    noise_mask = labels == -1
    cluster_mask = ~noise_mask

    # Split dfs
    noise_df = df_work.loc[noise_mask].drop(columns=['embedding_json']).reset_index(drop=True)
    clustered_df = df_work.loc[cluster_mask].drop(columns=['embedding_json']).reset_index(drop=True)

    # Extract only clustered embeddings
    X_clustered = X[cluster_mask]

    # Summary
    n_clusters = len(set(labels) - {-1})
    n_noise = int(noise_mask.sum())
    summary = {"n_clusters": n_clusters, "n_noise": n_noise}

    return X_clustered, clustered_df, noise_df


In [38]:
#X,CA_db , c_db = run_dbscan(CA_x2d)
X,CO_db, c_db = run_dbscan(CO_x2d)

In [9]:
def count_events_per_cluster(df: pd.DataFrame) -> pd.DataFrame:
    """
    Count how many times each GlobalEventID appears in each cluster.

    Args:
        df: DataFrame with columns ['GlobalEventID', 'cluster'].

    Returns:
        DataFrame with columns ['cluster', 'GlobalEventID', 'count'].
    """
    counts = (
        df.groupby(['cluster', 'GlobalEventID'])
          .size()
          .reset_index(name='count')
          .sort_values(['cluster', 'count'], ascending=[True, False])
    )
    return counts

In [14]:
def build_event_graph(df: pd.DataFrame, min_count: int = 1) -> nx.Graph:
    """
    Build a co-occurrence graph from ['cluster','GlobalEventID','count'].
    - Node attrs:
        total_count: sum of 'count' across clusters
        clusters: set of clusters the ID appears in
    - Edge attrs:
        weight: number of clusters the pair co-occurred in
        clusters: set of clusters where the pair co-occurred
    """
    G = nx.Graph()

    for cluster_id, group in df.groupby("cluster"):
        # Map event -> count within this cluster
        counts = dict(zip(group["GlobalEventID"], group["count"]))
        events = list(counts.keys())

        # Add/update nodes
        for ev, c in counts.items():
            if ev not in G:
                G.add_node(ev, total_count=0, clusters=set())
            G.nodes[ev]["total_count"] += int(c)
            G.nodes[ev]["clusters"].add(cluster_id)

        # Add/update edges for co-occurring pairs within this cluster
        for u, v in itertools.combinations(events, 2):
            if G.has_edge(u, v):
                G[u][v]["weight"] += 1
                G[u][v]["clusters"].add(cluster_id)
            else:
                G.add_edge(u, v, weight=1, clusters={cluster_id})

    # Convert sets to sorted lists for cleaner serialization/hover text
    for n, data in G.nodes(data=True):
        data["clusters"] = sorted(data["clusters"])
    for u, v, data in G.edges(data=True):
        data["clusters"] = sorted(data["clusters"])
    
        # 5) Remove edges that don’t meet min_count
    edges_to_remove = [(u, v) for u, v, d in G.edges(data=True) if d["weight"] < min_count]
    G.remove_edges_from(edges_to_remove)

    return G

In [10]:
count_CA = count_events_per_cluster(CA_db)
count_CO = count_events_per_cluster(CO_db)

In [11]:
count_CA

,cluster,GlobalEventID,count
0,0,298143273,2
1,0,298250095,2
2,0,298375880,2
3,1,298224407,6
4,2,298231070,2
5,2,298231154,2
6,2,298382471,2
7,2,298439944,2
8,3,298121840,1
9,3,298295924,1


In [15]:
G_CO = build_event_graph(count_CO)
G_CA = build_event_graph(count_CA)

In [17]:
print(f"Graph Built: {G_CO.number_of_nodes()} nodes, {G_CO.number_of_edges()} edges.")

Graph Built: 479 nodes, 6171 edges.


In [16]:
G_CA.nodes

NodeView((298143273, 298250095, 298375880, 298224407, 298231070, 298231154, 298382471, 298439944, 298121840, 298295924, 298533033, 298890753, 299111884, 298394818, 298636264, 298541219, 298633378, 298556473, 298629634, 298633395, 298620818, 298670845, 298671761, 298702788, 298730931, 298836521, 298847512, 298891299, 298742226, 298643776, 298645415))

In [22]:
M_CO = nx.to_numpy_array(G_CO)
M_CA = nx.to_numpy_array(G_CA)

In [23]:
M_CO

array([[0., 2., 2., ..., 0., 0., 0.],
       [2., 0., 2., ..., 0., 0., 0.],
       [2., 2., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 1., 0.]])

In [24]:
def spectral_embed_df(df: pd.DataFrame, n_components: int = 2, **kwargs) -> pd.DataFrame:
    """
    Run SpectralEmbedding on 'embedding_json' column and return a copy of df
    with new ['x','y', ...] columns. Rows with cluster == -1 are skipped
    and get NaN coordinates.
    
    Args:
        df : pd.DataFrame
            Must contain 'embedding_json' (string repr of list) and 'cluster'.
        n_components : int
            Number of embedding dimensions (default=2).
        **kwargs : passed to sklearn.manifold.SpectralEmbedding.
    """
    # Copy DataFrame
    df_copy = df.copy(deep=True)

    # Parse embedding_json into numpy arrays
    s = df_copy['embedding_json'].str.strip("[]").str.split(',')
    s = s.apply(lambda arr: np.array(arr, dtype=np.float32))
    df_copy['embedding_json'] = s

    # Mask out cluster == -1
    mask = df_copy["cluster"] != -1
    X = np.vstack(df_copy.loc[mask, "embedding_json"])

    # Run spectral embedding only on valid rows
    embedding = SpectralEmbedding(n_components=n_components, **kwargs)
    coords = embedding.fit_transform(X)

    # Build coordinate column names
    col_names = []
    if n_components >= 1:
        col_names.append("x")
    if n_components >= 2:
        col_names.append("y")
    for i in range(2, n_components):
        col_names.append(f"x{i}")

    # Initialize coordinate columns as NaN
    for col in col_names:
        df_copy[col] = pd.NA

    # Fill only valid rows
    for i, col in enumerate(col_names):
        df_copy.loc[mask, col] = coords[:, i]
    df_copy.drop(columns=['embedding_json'])
    return df_copy



In [25]:
def spectral_embed_from_adjacency(
    A: np.ndarray,
    n_components: int = 2,
    node_ids: Optional[Union[np.ndarray, list]] = None,
    drop_isolates: bool = False,
    make_symmetric: bool = False,
    zero_self_loops: bool = False,
    use_sparse: bool = False,
    **kwargs
) -> pd.DataFrame:
    """
    Compute a spectral embedding from an adjacency matrix and return a DataFrame
    with columns ['x','y', ...]. Works with dense or sparse input.

    Args:
        A : np.ndarray (n x n)
            Adjacency (non-negative). If not perfectly symmetric, set make_symmetric=True.
        n_components : int, default 2
            Dimensions of the embedding.
        node_ids : array-like of shape (n,), optional
            Labels to associate with rows. Defaults to range(n).
        drop_isolates : bool, default False
            If True, nodes with degree 0 are dropped from the output.
            If False, isolates are kept with NaN coordinates.
        make_symmetric : bool, default True
            If True, symmetrize A as (A + A.T) / 2.
        zero_self_loops : bool, default True
            If True, zero out the diagonal of A.
        use_sparse : bool, default True
            If True, pass a CSR matrix to sklearn (recommended for large graphs).
        **kwargs :
            Extra args passed to sklearn.manifold.SpectralEmbedding, e.g.:
            - eigen_solver={'arpack','amg','lobpcg'}
            - random_state=42
            Note: We set affinity='precomputed' internally.

    Returns:
        pd.DataFrame:
            Columns: ['node_id','x','y', ...] (or just coords if node_ids not provided).
            If drop_isolates=False, isolates have NaN for coordinates.
    """
    A = np.asarray(A)
    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("A must be a square (n x n) adjacency matrix.")

    n = A.shape[0]
    if node_ids is None:
        node_ids = np.arange(n)
    else:
        node_ids = np.asarray(node_ids)
        if node_ids.shape[0] != n:
            raise ValueError("node_ids must have length equal to A.shape[0].")

    # Basic hygiene on adjacency
    if make_symmetric:
        A = 0.5 * (A + A.T)
    if zero_self_loops:
        np.fill_diagonal(A, 0.0)

    # Identify isolates (degree 0)
    degrees = A.sum(axis=1)
    valid_mask = degrees > 0

    # Build the submatrix for non-isolates
    if valid_mask.any():
        A_valid = A[np.ix_(valid_mask, valid_mask)]
        if use_sparse:
            A_valid = sparse.csr_matrix(A_valid)

        # SpectralEmbedding expects an affinity matrix; adjacency is OK when using 'precomputed'
        emb = SpectralEmbedding(
            n_components=n_components,
            affinity="precomputed",
            **kwargs
        )
        coords_valid = emb.fit_transform(A_valid)  # (n_valid, n_components)
    else:
        # All isolates; produce an all-NaN coordinate matrix
        coords_valid = np.empty((0, n_components))

    # Build column names: x, y, x2, x3, ...
    col_names = []
    if n_components >= 1: col_names.append("x")
    if n_components >= 2: col_names.append("y")
    for i in range(2, n_components):
        col_names.append(f"x{i}")

    # Prepare output with NaNs, then fill for non-isolates
    out_coords = np.full((n, n_components), np.nan, dtype=float)
    out_coords[valid_mask, :] = coords_valid

    # Assemble DataFrame
    df_out = pd.DataFrame(out_coords, columns=col_names)
    df_out.insert(0, "node_id", node_ids)

    if drop_isolates:
        df_out = df_out.loc[valid_mask].reset_index(drop=True)

    return df_out


In [26]:
CA_emb = spectral_embed_from_adjacency(M_CA, n_components=2)


/Users/emilylugos/opt/anaconda3/lib/python3.9/site-packages/sklearn/manifold/_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


In [27]:
CA_emb

,node_id,x,y
0,0,0.030397,-0.023992
1,1,0.030397,-0.023992
2,2,0.030397,-0.023992
3,3,NaN,NaN
4,4,-0.007932,0.239983
5,5,-0.007932,0.239983
6,6,-0.007932,0.239983
7,7,-0.007932,0.239983
8,8,-0.018495,-0.011211
9,9,-0.018495,-0.011211


In [28]:
CO_emb = spectral_embed_from_adjacency(M_CO, n_components=2)


/Users/emilylugos/opt/anaconda3/lib/python3.9/site-packages/sklearn/manifold/_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


In [29]:

def plot_event_embedding(df: pd.DataFrame):
    """
    Plot a scatter plot of spectral embedding coordinates (x, y)
    with hover showing GlobalEventID and cluster.
    """
    fig = px.scatter(
        df,
        x="x",
        y="y",
        hover_data={"node_id": True, "x": False, "y": False},
        title="Spectral Embedding of Events",
        template="plotly_white",
        
    )
    fig.update_traces(marker=dict(size=6, opacity=0.7))
    return fig

In [30]:
fig = plot_event_embedding(CO_emb)
fig.show()

In [ ]:
#ran spectral
# now run clustering with just x & y?

SyntaxError: invalid syntax (1883632207.py, line 1)

In [75]:
CO_db

,GlobalEventID,date,source_index,subject,relation,object,sbert_text,cluster
0,1246818793,20250529,207,AI,generate,takeaways from article,AI generate takeaways from article,0
1,1246724796,20250529,91,Android users,can download,it,Android users can download it,1
2,1246750704,20250529,113,Aspen Public Radio,have joined NPR in,lawsuit,Aspen Public Radio have joined NPR in lawsuit,6
3,1246656131,20250529,52,Associated Press,contributed to,report,Associated Press contributed to report,2
4,1246819935,20250529,216,Colorado Attorney General,of,Office,Colorado Attorney General of Office,3
5,1246756951,20250529,122,Colorado Attorney General Phil Weiser,said in,statement,Colorado Attorney General Phil Weiser said in statement,4
6,1246646483,20240529,35,Colorado Cattlemen,’s,Association,Colorado Cattlemen ’s Association,5
7,1246750704,20250529,113,Colorado Public Radio,have joined NPR in,lawsuit,Colorado Public Radio have joined NPR in lawsuit,6
8,1246628216,20250529,22,Early symptoms,include,fever,Early symptoms include fever,7
9,1246714485,20250529,78,His brutal death,touched off,immediate protests,His brutal death touched off immediate protests,8
